Cellule 1 : **Imports et création des dossiers**

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Structure du projet (comme demandé dans la spec)
os.makedirs("data", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

pd.set_option("display.max_columns", None)
print("Pandas version :", pd.__version__)

Pandas version : 2.2.3


Cellule 2 : **Envoyer le fichier dans Colab**

In [2]:
from google.colab import files

uploaded = files.upload()   # choisissez data.csv

# Renommer et déplacer dans data/ pour respecter la structure
os.replace("data.csv", "data/raw_data.csv")
print(os.listdir("data"))

Saving data.csv to data.csv
['raw_data.csv']


Cellule 3 : **Chargement des données**

In [10]:
# Le fichier est encodé en ISO-8859-1 : sans ça, erreur UnicodeDecodeError
df = pd.read_csv("data/raw_data.csv", encoding="ISO-8859-1")

print("Dimensions :", df.shape)
df.head()

Dimensions : (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Cellule 4 : **Diagnostic initial**

In [4]:
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isna().sum())
print("\n--- Statistiques numériques ---")
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

--- Valeurs manquantes ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

--- Statistiques numériques ---
            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.6

Cellule 5 : **Doublons exacts**

In [5]:
nb_doublons = df.duplicated().sum()
print(f"Doublons exacts avant nettoyage : {nb_doublons}")

df = df.drop_duplicates().reset_index(drop=True)

print(f"Doublons après nettoyage : {df.duplicated().sum()}")
print("Nouvelles dimensions :", df.shape)

Doublons exacts avant nettoyage : 5268
Doublons après nettoyage : 0
Nouvelles dimensions : (536641, 8)


Cellule 6 : **Conversion des types (dates, ID, texte)**

In [6]:
# 1) Dates : texte -> datetime (format mois/jour/année heure:minute)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M")

# 2) CustomerID : c'est un identifiant, pas un nombre à calculer.
#    On le passe en texte (sans le ".0") en gardant les valeurs manquantes.
df["CustomerID"] = df["CustomerID"].astype("Int64").astype("string")

# 3) InvoiceNo et StockCode en texte
df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")

# Vérification stricte
print(df.dtypes)
assert pd.api.types.is_datetime64_any_dtype(df["InvoiceDate"])

InvoiceNo      string[python]
StockCode      string[python]
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID     string[python]
Country                object
dtype: object


Cellule 7 : **Anomalies métier (le cœur du jalon)**

  *7a. Explorer avant de décider*

In [7]:
# Factures d'annulation : commencent par "C"
df["is_cancellation"] = df["InvoiceNo"].str.startswith("C")

print("Lignes avec Quantity < 0 :", (df["Quantity"] < 0).sum())
print("  dont factures 'C' (retours) :", ((df["Quantity"] < 0) & df["is_cancellation"]).sum())
print("  dont SANS 'C' (anomalies)   :", ((df["Quantity"] < 0) & ~df["is_cancellation"]).sum())
print("Lignes avec UnitPrice == 0 :", (df["UnitPrice"] == 0).sum())
print("Lignes avec UnitPrice < 0  :", (df["UnitPrice"] < 0).sum())

# Regarder à quoi ressemblent les négatifs sans "C"
df[(df["Quantity"] < 0) & ~df["is_cancellation"]][["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]].head(10)

Lignes avec Quantity < 0 : 10587
  dont factures 'C' (retours) : 9251
  dont SANS 'C' (anomalies)   : 1336
Lignes avec UnitPrice == 0 : 2510
Lignes avec UnitPrice < 0  : 2


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
2362,536589,21777,NaN,-10,0.0
4286,536764,84952C,NaN,-38,0.0
7086,536996,22712,NaN,-20,0.0
7087,536997,22028,NaN,-20,0.0
7088,536998,85067,NaN,-6,0.0
7090,537000,21414,NaN,-22,0.0
7091,537001,21653,NaN,-6,0.0
7093,537003,85126,NaN,-2,0.0
7094,537004,21814,NaN,-30,0.0
7095,537005,21692,NaN,-70,0.0


  *7b. Codes qui ne sont pas des produits*

In [12]:
# StockCode non-produits : POST (frais de port), D (remise), M (manuel), BANK CHARGES, etc.
codes_speciaux = ["POST", "D", "M", "C2", "BANK CHARGES", "PADS", "DOT", "CRUK", "S", "B", "AMAZONFEE"]
print(df[df["StockCode"].isin(codes_speciaux)]["StockCode"].value_counts())

StockCode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
PADS               4
B                  3
Name: count, dtype: int64


  *7c. Décisions de traitement*

In [15]:
df["is_cancellation"] = df["InvoiceNo"].str.startswith("C")

# --- DÉCISION 1 : les retours (facture "C" + quantité négative) sont CONSERVÉS
#     et marqués. Ils sont de vraies informations métier (taux de retour, etc.).
df["is_return"] = df["is_cancellation"] & (df["Quantity"] < 0)

# --- DÉCISION 2 : les vraies anomalies sont ISOLÉES (pas supprimées en silence)
#     - Quantité négative sans facture "C" (casse, ajustement de stock)
#     - Prix <= 0 (ajustements comptables, cadeaux)
mask_anomalie = (
    ((df["Quantity"] < 0) & ~df["is_cancellation"])
    | (df["UnitPrice"] <= 0)
)
df_anomalies = df[mask_anomalie].copy()
df = df[~mask_anomalie].copy()

# --- DÉCISION 3 : lignes qui ne sont pas des produits -> isolées aussi
mask_non_produit = df["StockCode"].isin(codes_speciaux)
df_non_produits = df[mask_non_produit].copy()
df = df[~mask_non_produit].copy()

# Sauvegarder les lignes écartées pour la traçabilité
df_anomalies.to_csv("data/anomalies_ecartees.csv", index=False)
df_non_produits.to_csv("data/non_produits_ecartes.csv", index=False)

print(f"Anomalies isolées      : {len(df_anomalies)}")
print(f"Non-produits isolés    : {len(df_non_produits)}")
print(f"Retours conservés      : {df['is_return'].sum()}")
print(f"Lignes restantes       : {len(df)}")

Anomalies isolées      : 0
Non-produits isolés    : 2898
Retours conservés      : 8704
Lignes restantes       : 536494


Cellule 8 : **Standardisation du texte**

In [16]:
# Description : espaces en trop, majuscules, espaces multiples
df["Description"] = (
    df["Description"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# StockCode en majuscules
df["StockCode"] = df["StockCode"].str.strip().str.upper()

# Country : noms uniformisés
df["Country"] = df["Country"].str.strip()
corrections_pays = {
    "EIRE": "Ireland",
    "RSA": "South Africa",
    "USA": "United States",
    "Unspecified": pd.NA,
}
df["Country"] = df["Country"].replace(corrections_pays)

print(sorted(df["Country"].dropna().unique()))

['Australia', 'Austria', 'Bahrain', 'Belgium', 'Brazil', 'Canada', 'Channel Islands', 'Cyprus', 'Czech Republic', 'Denmark', 'European Community', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Iceland', 'Ireland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania', 'Malta', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Saudi Arabia', 'Singapore', 'South Africa', 'Spain', 'Sweden', 'Switzerland', 'United Arab Emirates', 'United Kingdom', 'United States']


Cellule 9 : **Valeurs manquantes restantes**

In [17]:
print(df.isna().sum())

# CustomerID manquant = clients "invités". On NE supprime PAS ces lignes
# (elles représentent des ventes réelles), on ajoute un indicateur.
df["has_customer_id"] = df["CustomerID"].notna()

# Descriptions manquantes restantes : très peu, inutilisables
df = df.dropna(subset=["Description"])
print("Lignes finales :", len(df))

InvoiceNo               0
StockCode               0
Description             0
Quantity                0
InvoiceDate             0
UnitPrice               0
CustomerID         131618
Country               446
is_cancellation         0
is_return               0
dtype: int64
Lignes finales : 536494


Cellule 10 : **Colonnes utiles pour la suite**

In [19]:
# Montant de la ligne (négatif pour les retours, ce qui est logique)
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

# S'assurer que InvoiceDate est bien de type datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M")

# Colonnes de date utiles pour les jalons suivants
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype("string")

# Retirer la colonne temporaire
df = df.drop(columns=["is_cancellation"])

df = df.reset_index(drop=True)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_return,has_customer_id,TotalPrice,Year,Month,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,True,15.30,2010,12,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,True,20.34,2010,12,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,True,22.00,2010,12,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,True,20.34,2010,12,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,True,20.34,2010,12,2010-12


Cellule 11 : **Validation des critères d'acceptation**

In [21]:
# Supprimer les doublons potentiellement réintroduits après standardisation
df = df.drop_duplicates().reset_index(drop=True)

assert df.duplicated().sum() == 0, "Il reste des doublons"
assert pd.api.types.is_datetime64_any_dtype(df["InvoiceDate"]), "Dates non converties"
assert (df.loc[~df["is_return"], "Quantity"] > 0).all(), "Quantités négatives hors retours"
assert (df["UnitPrice"] > 0).all(), "Prix <= 0 restants"

print("Tous les critères sont validés")
print(df.dtypes)
print("Dimensions finales :", df.shape)

Tous les critères sont validés
InvoiceNo                  object
StockCode                  object
Description                object
Quantity                    int64
InvoiceDate        datetime64[ns]
UnitPrice                 float64
CustomerID                float64
Country                    object
is_return                    bool
has_customer_id              bool
TotalPrice                float64
Year                        int32
Month                       int32
YearMonth          string[python]
dtype: object
Dimensions finales : (531237, 14)


Cellule 12 : **Export du fichier nettoyé**

In [22]:
df.to_csv("data/cleaned_data.csv", index=False)
print(os.listdir("data"))

# Télécharger sur votre PC
files.download("data/cleaned_data.csv")
files.download("data/anomalies_ecartees.csv")
files.download("data/non_produits_ecartes.csv")

['anomalies_ecartees.csv', 'non_produits_ecartes.csv', 'cleaned_data.csv', 'raw_data.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>